# Structured streaming Demo

In [1]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [ ]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [ ]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [ ]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [ ]:
from pyspark.sql.functions import window, col, count, avg, max as _max, sum as _sum, abs as _abs

events_df = parsed_df \
    .withColumn("event_time", col("timestamp").cast("timestamp")) \
    .withColumn("edit_size", col("length.new") - col("length.old")) \
    .withWatermark("event_time", "10 minutes")

In [ ]:
tumbling_df = events_df \
    .groupBy(window("event_time", "1 minute"), col("bot")) \
    .agg(count("*").alias("edit_count"),
         avg("edit_size").alias("avg_edit_size"),
         _max(_abs(col("edit_size"))).alias("max_abs_edit"))

tumbling_query = tumbling_df.writeStream \
    .queryName("tumbling") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .start()

In [ ]:
sliding_df = events_df \
    .groupBy(window("event_time", "5 minutes", "1 minute"), col("bot")) \
    .agg(count("*").alias("edit_count"),
         _sum(_abs(col("edit_size"))).alias("total_changed"))

sliding_query = sliding_df.writeStream \
    .queryName("sliding") \
    .format("console") \
    .outputMode("update") \
    .option("truncate", False) \
    .start()